# Music Generation with a Transformer: Next-Event Prediction

## Case study: Bach Chorales in a symbolic MIDI representation

In this notebook, we build a small decoder-only Transformer in PyTorch that learns to predict the next musical event. The goal is to demonstrate the full workflow: music representation, tokenization, training windows, positional encoding, a model with multiple inputs and outputs, teacher-forced training, and autoregressive sampling.

### Why is music generation challenging?

Music combines local dependencies (successive melody notes and rhythm) with long-range dependencies (motifs, phrases, cadences, and recurring harmony). MIDI stores a performance as timed events rather than prose. A model therefore needs to distinguish pitch, timing, voice, and sometimes simultaneous notes. This demonstration simplifies the problem to time steps and pitch classes.

### Autoregression and factorization

For a token sequence $x_1,\ldots,x_T$, the distribution is modeled as $p(x_{1:T})=\prod_t p(x_t\mid x_{<t})$. During training, the input is shifted by one step relative to the target. A causal mask prevents attention to future positions. During generation, the model appends one token at a time, so sampling is sequential.

### Transformer, position, and multiple outputs

Self-attention computes similarities between queries and keys, then combines values. Multi-head attention can learn different relationships in parallel. Attention alone has no notion of order, so we add sinusoidal positional encoding. In this example, each time step is a multidimensional pianoroll bit vector, and the output predicts all 48 pitch classes at once. This is a multi-output setup with independent Bernoulli logits, suitable for notes that can sound simultaneously.

### Loss function and limitations

We use binary cross-entropy per time step and pitch. Unlike softmax, which enforces exactly one class, sigmoid/BCE allows several notes to be active at once. A threshold of 0.5 produces a simple pianoroll; Bernoulli sampling can be used during generation. This is not a professional composition system: the short training run, small corpus, and simplified rhythmic grid are intended only to make the effect observable.

### Encoding polyphony

In a pianoroll representation, one time step can contain many notes, so chords do not require artificially ordering notes as single tokens. A real MIDI parser must pair note-on/note-off events, handle tempo, and quantize time. In practice, tokens can encode pitch, duration, velocity, and instrument, or use a multi-stream representation.

**Demonstration settings:** a small model (d_model=96, 3 layers, 4 heads), 16-step windows, and a few hundred updates. It should run in a reasonable time on both GPU and CPU; sound quality is not intended to represent production-level quality.

## 1. Data: JSB Chorales and the pianoroll representation

Bach Chorales is a classic small dataset of symbolic four-part scores. Each time step describes the pitches in the soprano, alto, tenor, and bass voices. In this case study, we use a simplified pianoroll representation: a `[time, pitch]` tensor where 1 indicates an active note. The loader first tries to download a CSV file from the JSB Chorales dataset. If the network or source URL is unavailable, it generates deterministic example harmony sequences; the cell reports which option was used. This lets you follow the full pipeline while making the demonstration fallback limitations explicit.

For the real dataset, we assume a CSV with cells containing lists of MIDI notes active at successive time steps. For stability, the example maps notes to 48 pitch classes (MIDI 48–95) and sequences to a fixed length. This is a simplification: note durations, dynamics, articulation, and precise timing are not encoded.

**What this cell does:** imports dependencies, sets random seeds, and builds a small example corpus of multi-voice pianorolls. You can enable an attempt to download JSB by setting `TRY_DOWNLOAD=True`.


In [ ]:
import math, random, urllib.request, csv, io
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED=23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",device)

PITCH_MIN, N_PITCH, STEPS = 48, 48, 32
TRY_DOWNLOAD=False  # Set to True to try downloading the public dataset.

def make_demo_corpus(n=240):
    # Four voices, diatonic progressions, and sustained rhythms: a controlled fallback.
    scales=[0,2,4,5,7,9,11]
    roots=[0,5,7,0,9,5,0,7]
    rows=[]
    for _ in range(n):
        roll=np.zeros((STEPS,N_PITCH),dtype=np.float32)
        trans=random.choice([-2,0,0,0,2])
        for t in range(STEPS):
            beat=t//4; root=roots[beat%len(roots)]
            chord=[root,(root+4)%12,(root+7)%12,(root+11)%12]
            pitches=[60+root+trans, 60+chord[1]+trans, 48+chord[2]+trans, 48+root+trans]
            for voice,p in enumerate(pitches):
                p=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,p))
                # Octave/pitch variation and occasional rests imitate variations between parts.
                q=p + (12 if voice<2 and random.random()<.25 else 0)
                q=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,q))
                if random.random()>.035: roll[t,q-PITCH_MIN]=1
        rows.append(roll)
    return np.stack(rows)

def load_jsb_or_demo():
    if TRY_DOWNLOAD:
        urls=["https://raw.githubusercontent.com/czhuang/JSB-Chorales-dataset/master/jsb-chorales-train.csv"]
        for url in urls:
            try:
                raw=urllib.request.urlopen(url,timeout=8).read().decode("utf-8")
                # This public format stores one observation per row.
                parsed=[]; sequence=[]
                for row in csv.reader(io.StringIO(raw)):
                    vals=[int(x) for x in row if x.strip().lstrip('-').isdigit()]
                    if not vals:
                        if len(sequence)>=STEPS:
                            for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                                parsed.append(np.stack(sequence[start:start+STEPS]))
                        sequence=[]; continue
                    frame=np.zeros(N_PITCH,dtype=np.float32)
                    for p in vals:  # Multiple voices/notes may be active in one time step.
                        if PITCH_MIN<=p<PITCH_MIN+N_PITCH: frame[p-PITCH_MIN]=1
                    sequence.append(frame)
                if len(sequence)>=STEPS:
                    for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                        parsed.append(np.stack(sequence[start:start+STEPS]))
                if len(parsed)>20: return np.stack(parsed),"downloaded JSB Chorales (CSV; polyphonic time steps)"
            except Exception as e: print("Download failed:",type(e).__name__,str(e)[:120])
    return make_demo_corpus(),"demonstration fallback (synthetic harmonies)"

rolls,data_source=load_jsb_or_demo()
print("Source:",data_source,"| number of sequences:",len(rolls),"| shape:",rolls.shape)
plt.figure(figsize=(10,2.5)); plt.imshow(rolls[0].T,aspect="auto",origin="lower",cmap="magma"); plt.xlabel("step"); plt.ylabel("MIDI pitch (index)" ); plt.title("Example pianoroll"); plt.show()


## 2. Preparing sequences and splitting the data

We split sequences into train and validation sets by complete examples, so windows from the same piece do not appear in both splits. We then train on short input windows, with the target pianoroll shifted by one step. This is teacher forcing: the model receives the true history during training.

**What this cell does:** creates the split, converts data to tensors, and samples a random batch of inputs and targets. With larger datasets, preserve the original train/validation/test split and keep a separate test set.


In [ ]:
perm=np.random.permutation(len(rolls)); cut=max(1,int(.85*len(rolls)))
train_np, val_np=rolls[perm[:cut]], rolls[perm[cut:]]
train=torch.tensor(train_np,device=device); val=torch.tensor(val_np,device=device)
CONTEXT=16; BATCH=24

def get_batch(split):
    data=train if split=="train" else val
    if len(data)==0: data=train
    ids=torch.randint(0,len(data),(BATCH,),device=device)
    starts=torch.randint(0,STEPS-CONTEXT-1,(BATCH,),device=device)
    x=torch.stack([data[i,s:s+CONTEXT] for i,s in zip(ids,starts)])
    y=torch.stack([data[i,s+1:s+CONTEXT+1] for i,s in zip(ids,starts)])
    return x,y
xb,yb=get_batch("train")
print("train/validation:",len(train),len(val),"batch:",xb.shape,"target:",yb.shape)


## 3. Position and causal self-attention

Sinusoidal positional encoding adds the step index to the feature representation. In multi-head attention, each head projects the input into queries, keys, and values. Before softmax, the lower-triangular causal mask sets future positions to a very negative value, making their attention weights zero.

**What this cell does:** implements sinusoidal encoding, multi-head attention with a causal mask, and a pre-norm block with a residual connection. This exposes the Transformer components instead of hiding them behind a ready-made library.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=256):
        super().__init__(); pos=torch.arange(max_len).float().unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_len,d_model); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class CausalMHA(nn.Module):
    def __init__(self,d_model,heads,drop=.1):
        super().__init__(); assert d_model%heads==0
        self.h=heads; self.dh=d_model//heads
        self.qkv=nn.Linear(d_model,3*d_model); self.proj=nn.Linear(d_model,d_model); self.drop=nn.Dropout(drop)
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.qkv(x).chunk(3,dim=-1)
        def split(z): return z.view(B,T,self.h,self.dh).transpose(1,2)
        q,k,v=map(split,(q,k,v)); scores=(q@k.transpose(-2,-1))/math.sqrt(self.dh)
        mask=torch.triu(torch.ones(T,T,device=x.device,dtype=torch.bool),diagonal=1)
        scores=scores.masked_fill(mask,float("-inf")); a=self.drop(scores.softmax(-1))
        out=(a@v).transpose(1,2).contiguous().view(B,T,C)
        return self.proj(out)

class TransformerBlock(nn.Module):
    def __init__(self,d_model,heads,drop=.1):
        super().__init__(); self.n1=nn.LayerNorm(d_model); self.attn=CausalMHA(d_model,heads,drop)
        self.n2=nn.LayerNorm(d_model); self.ff=nn.Sequential(nn.Linear(d_model,4*d_model),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d_model,d_model),nn.Dropout(drop))
    def forward(self,x):
        x=x+self.attn(self.n1(x)); return x+self.ff(self.n2(x))


## 4. Model: multidimensional input and output

Each pianoroll step is a 48-dimensional input vector. A linear projection maps it to `d_model`. The Transformer stack processes the sequence, and the output head returns 48 logits per step. We do not use token embeddings because the input is not a single token ID, but a set of simultaneous notes.

**What this cell does:** defines the model and checks the output shape. The `[batch, time, pitch]` output matches the binary pianoroll target.


In [ ]:
class MusicTransformer(nn.Module):
    def __init__(self,n_pitch=N_PITCH,d_model=96,heads=4,layers=3,drop=.1,max_len=128):
        super().__init__(); self.in_proj=nn.Linear(n_pitch,d_model); self.pos=SinusoidalPositionalEncoding(d_model,max_len)
        self.blocks=nn.ModuleList([TransformerBlock(d_model,heads,drop) for _ in range(layers)])
        self.norm=nn.LayerNorm(d_model); self.out=nn.Linear(d_model,n_pitch)
    def forward(self,x):
        z=self.pos(self.in_proj(x))
        for block in self.blocks: z=block(z)
        return self.out(self.norm(z))

model=MusicTransformer().to(device)
print("Parameters:",sum(p.numel() for p in model.parameters()))
print("logits:",model(xb).shape)


## 5. Training and metrics

We optimize BCE-with-logits, which combines sigmoid and binary cross-entropy in a numerically stable way. We also report activity (the share of predicted notes above a 0.5 threshold), because a model can achieve a deceptively low loss by predicting mostly silence. The short validation loss is a demonstration metric, not an objective measure of composition quality.

**What this cell does:** runs a bounded training loop with gradient clipping and plots train/validation loss over the updates. Reduce `STEPS_TRAIN` on a slow CPU; increase it only after checking runtime.


In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
STEPS_TRAIN=240; EVAL_EVERY=40
history={"step":[],"train":[],"val":[]}
@torch.no_grad()
def eval_loss(n_batches=8):
    model.eval(); vals=[]
    for _ in range(n_batches):
        x,y=get_batch("val"); vals.append(F.binary_cross_entropy_with_logits(model(x),y).item())
    model.train(); return float(np.mean(vals))
for step in range(1,STEPS_TRAIN+1):
    x,y=get_batch("train"); logits=model(x); loss=F.binary_cross_entropy_with_logits(logits,y)
    optimizer.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
    if step%EVAL_EVERY==0 or step==1:
        vl=eval_loss(); history["step"].append(step); history["train"].append(loss.item()); history["val"].append(vl)
        print(f"step {step:3d} | train BCE {loss.item():.4f} | validation BCE {vl:.4f}")
plt.plot(history["step"],history["train"],marker="o",label="train"); plt.plot(history["step"],history["val"],marker="o",label="validation")
plt.xlabel("update"); plt.ylabel("BCE"); plt.grid(); plt.legend(); plt.show()


## 6. Generation and pianoroll analysis

During generation, we provide an initial fragment, take the prediction for the next step, and append it to the context. Here we use Bernoulli sampling from sigmoid probabilities. `temperature` scales the logits: a lower value makes predictions more decisive, while a higher value increases uncertainty. We also show activity and a simple pianoroll plot.

**What this cell does:** autoregressively generates several steps and compares the real context with the generated continuation. Since training is short, the result may sound chaotic; the main goal is to follow the mechanics.


In [ ]:
@torch.no_grad()
def generate(seed_roll,steps=16,temperature=1.0,threshold=None):
    model.eval(); context=torch.tensor(seed_roll[:CONTEXT],device=device).unsqueeze(0)
    generated=[]
    for _ in range(steps):
        logits=model(context[:,-CONTEXT:])[:,-1,:]/temperature
        probs=logits.sigmoid()
        nxt=torch.bernoulli(probs) if threshold is None else (probs>threshold).float()
        generated.append(nxt[0].cpu().numpy()); context=torch.cat([context,nxt[:,None,:]],dim=1)
    model.train(); return np.stack(generated)
seed=val_np[0] if len(val_np) else train_np[0]
gen=generate(seed,steps=16,temperature=.85)
fig,ax=plt.subplots(1,2,figsize=(11,3))
ax[0].imshow(seed[:16].T,aspect="auto",origin="lower",cmap="magma"); ax[0].set_title("Data context")
ax[1].imshow(gen.T,aspect="auto",origin="lower",cmap="magma"); ax[1].set_title("Model continuation")
for a in ax: a.set_xlabel("step"); a.set_ylabel("pitch index")
plt.tight_layout(); plt.show()
print("Mean active notes per step:",round(float(gen.sum(1).mean()),2))


## Key takeaways

- A Transformer is an autoregressive conditional model: it predicts the next step from the history so far.
- The causal attention mask prevents information from the future from leaking in; positional encoding supplies order information.
- A pianoroll encodes polyphony through multiple active pitches at the same step. BCE-with-logits suits multiple independent labels.
- Teacher-forced training is parallel across positions, while generation remains sequential.
- Short training on a simplified or synthetic corpus demonstrates the mechanics, not composition quality. Comparisons should state the data source, split, representation, and sampling method.
- Possible extensions: MIDI parsing (note-on/off, tempo, instrument), duration and velocity tokens, longer context, key control, and evaluation on separate test pieces.